## IMPORTACION DE LIBRERIAS

In [1]:
import pandas as pd
import numpy as np

import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score,
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV


import matplotlib.pyplot as plt

from imblearn.over_sampling import SMOTE
from collections import Counter

In [2]:
bucket_name = "datospco"

datos = pd.read_csv(f"s3://{bucket_name}/{'datos_train.csv'}")

In [3]:
datos["estrato_social"] = datos["estrato_social"].astype(str)

datos["target"] = (datos["categoria"] == "TECNOLOGIA").astype(int)
datos.drop(columns="categoria", inplace=True)

In [4]:
datos["target"].value_counts()

target
0    19740
1      433
Name: count, dtype: int64

In [5]:
datos = datos[
    [
        "tipo_transaccion",
        "valor_transaccion",
        "id_aliado",
        "genero",
        "estrato_social",
        "estado_civil",
        "ocupacion",
        "ciudad",
        "target",
    ]
]

In [6]:
X = datos.drop(columns=["target"])  # variables predictoras
y = datos["target"]

In [7]:
num_vars = ["valor_transaccion"]
cat_vars = [
    "tipo_transaccion",
    "id_aliado",
    "genero",
    "estrato_social",
    "estado_civil",
    "ocupacion",
    "ciudad",
]

In [8]:
# Transformaciones
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_vars),
        ('cat', categorical_transformer, cat_vars)
    ])

In [9]:
# Dividir en entrenamiento y validación
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [10]:
# print("Distribución original:", Counter(y_train))
# smote = SMOTE(random_state=42, sampling_strategy=0.5)
# X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
# print("Distribución balanceada:", Counter(y_resampled))

In [11]:
X_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4035 entries, 18699 to 8635
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   tipo_transaccion   4035 non-null   object 
 1   valor_transaccion  4035 non-null   float64
 2   id_aliado          4035 non-null   int64  
 3   genero             4035 non-null   object 
 4   estrato_social     4035 non-null   object 
 5   estado_civil       4035 non-null   object 
 6   ocupacion          4035 non-null   object 
 7   ciudad             4035 non-null   object 
dtypes: float64(1), int64(1), object(6)
memory usage: 283.7+ KB


In [12]:
X_test.iloc[0]

tipo_transaccion            Sale
valor_transaccion        59000.0
id_aliado                     21
genero                         F
estrato_social               6.0
estado_civil              Casado
ocupacion            Ama de Casa
ciudad                    BOGOTA
Name: 18699, dtype: object

In [19]:
# ⚡ Entrenamiento de Logistic Regression con preprocesamiento
def train_logisticR(X_train, y_train, preprocessor):
    # Definir pipeline: preprocesamiento + modelo
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ])

    # Definir grilla de hiperparámetros (usando prefijo classifier__)
    param_grid = {
        "classifier__C": [0.01, 0.1, 1, 10],
        "classifier__penalty": ["l2"],  # L1/L2 dependen del solver
        "classifier__solver": ["liblinear", "saga"],
        "classifier__class_weight": [None, "balanced"],
    }

    # GridSearchCV
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="roc_auc",  # o "f1", depende del problema
        cv=5,
        n_jobs=-1,
        verbose=2,
    )

    # Entrenar con búsqueda de hiperparámetros
    grid_search.fit(X_train, y_train)

    print("Mejores parámetros Logistic Regression:", grid_search.best_params_)

    # Retornar el mejor pipeline (preprocesamiento + modelo entrenado)
    return grid_search.best_estimator_


In [20]:
# Función para entrenar Random Forest
def trainRFclass(X_train, y_train):
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(class_weight="balanced_subsample", random_state=42))
    ])

    param_grid = {
        "classifier__n_estimators": [100, 200, 300],
        "classifier__max_depth": [3, 5, 7, None],
        "classifier__min_samples_split": [2, 5, 10],
        "classifier__min_samples_leaf": [1, 2, 5, 10],
        "classifier__max_features": ["sqrt", "log2"],
    }

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="f1",
        cv=3,
        n_jobs=-1,
        verbose=2,
    )

    grid_search.fit(X_train, y_train)
    print("Mejores parámetros RF:", grid_search.best_params_)

    return grid_search.best_estimator_

In [21]:
def encontrar_umbral(y_proba, y_test):
    umbral_optimo = 0
    mejor_f1 = 0
    for t in np.arange(0.01, 1, 0.01):
        y_pred_bin = (y_proba >= t).astype(int)
        f1 = f1_score(y_test, y_pred_bin)
        if f1 > mejor_f1:
            mejor_f1 = f1
            umbral_optimo = t
    print(f"Umbral óptimo: {umbral_optimo}, F1: {mejor_f1}")

    return umbral_optimo

## Regresion logistica

In [23]:
modelLR = train_logisticR(X_train, y_train, preprocessor)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


/home/sagemaker-user/.conda/envs/py310/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/sagemaker-user/.conda/envs/py310/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/sagemaker-user/.conda/envs/py310/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/sagemaker-user/.conda/envs/py310/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/sagemaker-user/.conda/envs/py310/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  

Mejores parámetros Logistic Regression: {'classifier__C': 10, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear'}


/home/sagemaker-user/.conda/envs/py310/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [31]:
y_probaRF = best_rf_pipeline.predict_proba(X_test)[:, 1]

umbral_optimoRF = encontrar_umbral(y_probaRF, y_test)

# 8. Predicciones finales con ese umbral
y_pred_final = (y_probaRF >= 0.4).astype(int)

print("Matriz de confusión:\n", confusion_matrix(y_test, y_pred_final))
print("Reporte de clasificación:\n", classification_report(y_test, y_pred_final))

Umbral óptimo: 0.7100000000000001, F1: 0.4246575342465754
Matriz de confusión:
 [[3804  144]
 [  42   45]]
Reporte de clasificación:
               precision    recall  f1-score   support

           0       0.99      0.96      0.98      3948
           1       0.24      0.52      0.33        87

    accuracy                           0.95      4035
   macro avg       0.61      0.74      0.65      4035
weighted avg       0.97      0.95      0.96      4035



## Random forest

In [26]:
best_rf_pipeline = trainRFclass(X_train, y_train)

Fitting 3 folds for each of 288 candidates, totalling 864 fits
[CV] END classifier__C=0.01, classifier__class_weight=None, classifier__penalty=l2, classifier__solver=liblinear; total time=   0.2s
[CV] END classifier__C=0.01, classifier__class_weight=None, classifier__penalty=l2, classifier__solver=liblinear; total time=   0.2s
[CV] END classifier__C=0.01, classifier__class_weight=None, classifier__penalty=l2, classifier__solver=saga; total time=   7.8s
[CV] END classifier__C=0.01, classifier__class_weight=None, classifier__penalty=l2, classifier__solver=saga; total time=   4.4s
[CV] END classifier__C=0.01, classifier__class_weight=balanced, classifier__penalty=l2, classifier__solver=liblinear; total time=   0.1s
[CV] END classifier__C=0.01, classifier__class_weight=balanced, classifier__penalty=l2, classifier__solver=liblinear; total time=   0.1s
[CV] END classifier__C=0.01, classifier__class_weight=balanced, classifier__penalty=l2, classifier__solver=liblinear; total time=   0.1s
[CV]

In [25]:
y_probaLR = modelLR.predict_proba(X_test)[:, 1]

umbral_optimo_LR = encontrar_umbral(y_probaLR, y_test)

# 8. Predicciones finales con ese umbral
y_pred_final = (y_probaLR >= 0.4).astype(int)

print("Matriz de confusión:\n", confusion_matrix(y_test, y_pred_final))
print("Reporte de clasificación:\n", classification_report(y_test, y_pred_final))

Umbral óptimo: 0.7000000000000001, F1: 0.1605839416058394
Matriz de confusión:
 [[2628 1320]
 [  15   72]]
Reporte de clasificación:
               precision    recall  f1-score   support

           0       0.99      0.67      0.80      3948
           1       0.05      0.83      0.10        87

    accuracy                           0.67      4035
   macro avg       0.52      0.75      0.45      4035
weighted avg       0.97      0.67      0.78      4035



In [27]:
joblib.dump(best_rf_pipeline, "../despliegue/rf_pipeline.joblib")

['../despliegue/rf_pipeline.joblib']

In [19]:
import joblib
import os

# Nombre del archivo dentro de la carpeta
file_path = os.path.join("../despliegue", "modelRF.joblib")

# Guardar el modelo
joblib.dump(modelRF, file_path)

print(f"Modelo guardado en: {file_path}")

Modelo guardado en: ../despliegue/modelRF.joblib
